## Summary & Next Steps

### What We Covered

1. **System Architecture**: 4-agent design for climate intelligence
2. **Satellite Analysis**: Land-use classification with CNN
3. **Climate Data Integration**: Multi-source data aggregation
4. **LLM Reasoning**: Natural language question answering
5. **Policy Recommendations**: Evidence-based interventions
6. **Explainability**: Grad-CAM visualization
7. **Time-Series Analysis**: Trend detection and anomalies
8. **Agent Orchestration**: Multi-agent workflows

###  Next Steps

#### Immediate Implementation
- [ ] Configure Google Earth Engine credentials
- [ ] Set up OpenAI/Anthropic API keys
- [ ] Load real Sentinel-2 satellite imagery
- [ ] Connect to COPERNICUSData Hub for satellite data
- [ ] Integrate EDGAR/CAMS for real emissions data

#### Advanced Features
- [ ] Fine-tune satellite classification model on domain data
- [ ] Add multi-temporal change detection (U-Net)
- [ ] Implement real LLM chain with memory
- [ ] Build REST API for production deployment
- [ ] Create web dashboard for visualization
- [ ] Add custom vector embeddings for domain-specific search

#### Production Deployment
- [ ] Containerize with Docker
- [ ] Set up Kubernetes orchestration
- [ ] Implement monitoring & logging
- [ ] Create CI/CD pipeline
- [ ] Scale for real-time satellite processing

### Resources
- **Notebook Usage**: `python -m jupyter notebook`
- **CLI Tool**: `python main.py ask --question "..."`
- **Configuration**: Edit `config/settings.yaml`
- **Documentation**: See `README.md` and `DEVELOPMENT.md`

### Questions?
Review the agent implementations in `src/agents/` for customization options!

In [ ]:
print("Interactive Climate Intelligence Query System\n")
print("=" * 80)

# Example queries demonstrating different agent combinations
example_queries = [
    {
        'query': "What is the extent of deforestation in Irish regions?",
        'context': {'region': 'Ireland', 'temporal_range': [2020, 2024]},
        'expected_agents': ['Satellite', 'Reasoning']
    },
    {
        'query': "What are the policy solutions for rising emissions in Ireland?",
        'context': {'region': 'Ireland'},
        'expected_agents': ['Data', 'Reasoning', 'Policy']
    },
    {
        'query': "How should Ireland protect remaining biodiversity?",
        'context': {'region': 'Irish biodiversity hotspots'},
        'expected_agents': ['Data', 'Reasoning', 'Policy']
    }
]

# Run multiple queries with different agent paths
for idx, query_spec in enumerate(example_queries[:2], 1):
    print(f"\n Query {idx}: {query_spec['query']}")
    print(f"   Expected Agents: {' → '.join(query_spec['expected_agents'])}")
    print("   " + "-" * 76)
    
    response = copilot.ask(
        question=query_spec['query'],
        context=query_spec['context']
    )
    
    # Show activated agents
    activated = []
    if response.satellite_analysis:
        activated.append(' Satellite')
    if response.climate_data:
        activated.append('Data')
    if response.reasoning:
        activated.append(' Reasoning')
    if response.policy_recommendations:
        activated.append('Policy')
    
    print(f"  Activated: {' → '.join(activated)}")
    
    # Show key insights
    if response.reasoning and response.reasoning.success:
        print(f"   Insight: {response.reasoning.data.get('explanation', 'N/A')[:70]}...")
    
    if response.policy_recommendations and response.policy_recommendations.success:
        recs = response.policy_recommendations.data.get('recommendations', [])
        if recs:
            print(f"   Top Policy: {recs[0]['policy']} (Priority #{recs[0]['priority_rank']})")

print("\n" + "=" * 80)
print(" Interactive query system demonstration complete!")
print("\n Tips:")
print("   • The copilot automatically selects relevant agents based on keywords")
print("   • Satellite, Data, Reasoning, and Policy agents work together")
print("   • Results are cached for efficiency")
print("   • Use `copilot.ask(\"your question\")` for custom queries")

## 11. Interactive Query System

Query the copilot with natural language questions and see full agent orchestration in action.

In [ ]:
from data import ClimateDataProcessor

print(" Time-Series Change Detection Analysis\n")

# Create synthetic time-series data
years = list(range(2015, 2025))
# Simulated deforestation trend (decreasing forest cover)
forest_cover = np.array([85, 84, 83, 82, 80, 78, 76, 74, 72, 70])
emissions = np.array([120, 125, 128, 130, 135, 140, 145, 148, 150, 152])
temperature_anomaly = np.array([-0.5, -0.3, 0.1, 0.4, 0.6, 0.8, 1.0, 1.2, 1.4, 1.5])

print(" Analyzing Trends:\n")

# 1. Forest Cover Trend
forest_trend = ClimateDataProcessor.calculate_trends(years, forest_cover.tolist())
print(f" Forest Cover Trend:")
print(f"   Direction: {forest_trend['trend_direction']}")
print(f"   Change: {forest_trend['slope']:.2f}% per year")
print(f"   Over decade: {forest_trend['annual_change'] * 10:.1f}%")
print(f"   R-squared: {forest_trend['r_squared']:.3f}")

# 2. Emissions Trend
emissions_trend = ClimateDataProcessor.calculate_trends(years, emissions.tolist())
print(f"\n Emissions Trend:")
print(f"   Direction: {emissions_trend['trend_direction']}")
print(f"   Change: {emissions_trend['slope']:.2f} units/year")
print(f"   Decade change: {emissions_trend['annual_change'] * 10:.1f} units")

# 3. Temperature Anomaly Trend
temp_trend = ClimateDataProcessor.calculate_trends(years, temperature_anomaly.tolist())
print(f"\n Temperature Anomaly Trend:")
print(f"   Direction: {temp_trend['trend_direction']}")
print(f"   Warming rate: {temp_trend['slope']:.3f}°C per year")
print(f"   Total warming: {temp_trend['annual_change'] * 10:.1f}°C (2015-2024)")

# Extreme events detection
print(f"\nExtreme Events Detection:")
emission_extremes = ClimateDataProcessor.detect_extreme_events(emissions, threshold_percentile=90)
print(f"   Years with extreme emissions: {[years[i] for i in emission_extremes['extreme_indices']]}")
print(f"   Frequency: {emission_extremes['frequency']:.1%}")

# Visualization
fig, axes = plt.subplots(1, 3, figsize=(15, 4))
fig.suptitle('Time-Series Change Detection (2015-2024)', fontsize=12, fontweight='bold')

# Forest Cover
ax = axes[0]
ax.plot(years, forest_cover, 'o-', linewidth=2, markersize=6, color='green', label='Observed')
z = np.polyfit(years, forest_cover, 1)
p = np.poly1d(z)
ax.plot(years, p(years), '--', color='red', alpha=0.7, label=f'Trend (slope={z[0]:.2f})')
ax.set_xlabel('Year')
ax.set_ylabel('Forest Cover (%)')
ax.set_title('Forest Cover Decline')
ax.legend()
ax.grid(alpha=0.3)

# Emissions
ax = axes[1]
ax.plot(years, emissions, 'o-', linewidth=2, markersize=6, color='red', label='Observed')
z = np.polyfit(years, emissions, 1)
p = np.poly1d(z)
ax.plot(years, p(years), '--', color='darkred', alpha=0.7, label=f'Trend (slope={z[0]:.2f})')
ax.axhline(y=np.percentile(emissions, 90), color='orange', linestyle=':', label='90th percentile')
ax.set_xlabel('Year')
ax.set_ylabel('Emissions (Gt CO₂eq)')
ax.set_title('Emissions Increase')
ax.legend()
ax.grid(alpha=0.3)

# Temperature Anomaly
ax = axes[2]
colors = ['blue' if t < 0 else 'red' for t in temperature_anomaly]
ax.bar(years, temperature_anomaly, color=colors, alpha=0.7, label='Anomaly')
z = np.polyfit(years, temperature_anomaly, 1)
p = np.poly1d(z)
ax.plot(years, p(years), '--', color='darkred', linewidth=2, label=f'Trend ({z[0]:.3f}°C/yr)')
ax.axhline(y=0, color='black', linestyle='-', linewidth=0.5)
ax.set_xlabel('Year')
ax.set_ylabel('Temperature Anomaly (°C)')
ax.set_title('Warming Trend')
ax.legend()
ax.grid(alpha=0.3, axis='y')

plt.tight_layout()
plt.show()

print("\n Change detection analysis complete!")

## 10. Time-Series Change Detection

Analyze temporal trends in satellite imagery and climate data.

In [ ]:
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches

print(" Satellite Image Explainability Analysis\n")

# Create visualization
fig, axes = plt.subplots(2, 2, figsize=(12, 10))
fig.suptitle('Satellite Analysis: Land-Use Classification & Explainability', fontsize=14, fontweight='bold')

# 1. RGB Composite
ax = axes[0, 0]
ax.imshow(rgb, extent=[bbox['min_lon'], bbox['max_lon'], bbox['min_lat'], bbox['max_lat']])
ax.set_title('RGB Composite (Sentinel-2 B4,B3,B2)')
ax.set_xlabel('Longitude')
ax.set_ylabel('Latitude')

# 2. NDVI (Vegetation Index)
ax = axes[0, 1]
im = ax.imshow(ndvi, cmap='RdYlGn', vmin=-1, vmax=1)
ax.set_title('NDVI: Vegetation Health')
ax.set_xlabel('Pixel X')
ax.set_ylabel('Pixel Y')
plt.colorbar(im, ax=ax, label='NDVI')

# 3. NDBI (Built-up Index)
ax = axes[1, 0]
im = ax.imshow(ndbi, cmap='gray')
ax.set_title('NDBI: Urban/Built-up Areas')
ax.set_xlabel('Pixel X')
ax.set_ylabel('Pixel Y')
plt.colorbar(im, ax=ax, label='NDBI')

# 4. Classification Result
ax = axes[1, 1]
ax.axis('off')

# Show classification results
classification_text = f"""
LAND-USE CLASSIFICATION RESULT

Primary Class: {result.data['classification']['land_use_class']}
Confidence: {result.data['classification']['confidence']:.1%}

Class Probabilities:
"""

for i, (land_class, prob) in enumerate(
    list(result.data['classification']['all_probabilities'].items())[:5]
):
    bar_width = prob
    classification_text += f"\n{land_class:15} {prob:.1%} "
    # Simple text bar
    classification_text += "█" * int(prob * 30)

classification_text += f"\n\nGrad-CAM Visualization:"
classification_text += f"\n{result.data['explainability']['note']}"

ax.text(0.05, 0.95, classification_text, transform=ax.transAxes,
        fontsize=10, verticalalignment='top', family='monospace',
        bbox=dict(boxstyle='round', facecolor='wheat', alpha=0.5))

plt.tight_layout()
plt.show()

print(" Visualization complete!")

## 9. Explainability with Grad-CAM

Visualize which regions of satellite images drive predictions.

In [ ]:
from agents import ClimateCopilot

print(" Initializing Climate Intelligence Copilot...")

copilot = ClimateCopilot(config.to_dict())

print(" Copilot ready!\n")

# Ask a complex multi-agent question
question = "Which regions in Ireland have rising land degradation and why?"

print(f" Question: {question}\n")
print(" Orchestrating agents...\n")

response = copilot.ask(
    question=question,
    context={
        'region': 'Ireland',
        'temporal_range': [2020, 2024]
    }
)

print("=" * 80)
print("Multi-Agent Analysis Results")
print("=" * 80)

# Show which agents were activated
print(f"\n Agents Activated:")
agents_used = []
if response.satellite_analysis:
    print(f"   Satellite Agent: ACTIVE")
    agents_used.append('Satellite')
if response.climate_data:
    print(f"   Data Agent: ACTIVE")
    agents_used.append('Data')
if response.reasoning:
    print(f"   Reasoning Agent: ACTIVE")
    agents_used.append('Reasoning')
if response.policy_recommendations:
    print(f"   Policy Agent: ACTIVE")
    agents_used.append('Policy')

print(f"\nAgent Pipeline: {' → '.join(agents_used)}")

# Show integrated results
print(f"\n Integrated Analysis:")
if response.reasoning and response.reasoning.success:
    reasoning_data = response.reasoning.data
    print(f"  Question Type: {reasoning_data.get('question_type', 'N/A')}")
    print(f"  Main Explanation: {reasoning_data.get('explanation', 'N/A')[:100]}...")

if response.policy_recommendations and response.policy_recommendations.success:
    policy_data = response.policy_recommendations.data
    print(f"\n Policy Insight:")
    print(f"  Issue Type: {policy_data['issue_type']}")
    print(f"  Severity: {policy_data['severity']}")
    if policy_data.get('recommendations'):
        top_rec = policy_data['recommendations'][0]
        print(f"  Top Recommendation: {top_rec['policy']}")

print("\n" + "=" * 80)

## 8. Multi-Agent Orchestration

Use ClimateCopilot to orchestrate all agents for comprehensive climate intelligence.

In [ ]:
from agents.policy_agent import PolicyAgent

print(" Initializing Policy Agent...")

policy_agent = PolicyAgent(config.get('agents.policy', {}))

# Generate policy recommendations for deforestation
policy_input = {
    'issue_type': 'deforestation',
    'region': 'Ireland',
    'severity': 'high',
    'context': {
        'satellite_data': satellite_input,
        'climate_data': emissions_data if emissions_result.success else {}
    }
}

print("Generating policy recommendations for deforestation...\n")

policy_result = policy_agent.run(policy_input)

if policy_result.success:
    data = policy_result.data
    
    print(f" Policy Analysis for {data['issue_type']} (Severity: {data['severity']})\n")
    
    # Show top recommendations
    print("Top Policy Recommendations (by priority):\n")
    for rec in data['recommendations'][:5]:
        print(f"  Rank #{rec['priority_rank']}: {rec['policy']}")
        print(f"    Priority Score: {rec['priority_score']:.2f}")
        print(f"    Impact: {rec['impact_score']:.2f}  |  Feasibility: {rec['feasibility_score']:.2f}")
        print(f"    Cost: {rec['cost_category']}")
        print(f"    Co-benefits: {', '.join(rec['co_benefits'][:2])}")
        print()
    
    # Show timeline
    print(" Implementation Timeline:")
    timeline = data['implementation_timeline']
    print(f"  Immediate (0-6 months): {timeline['immediate_0_6_months']}")
    print(f"  Short-term (6-24 months): {timeline['short_term_6_24_months']}")
    print(f"  Medium-term (2-5 years): {timeline['medium_term_2_5_years']}")
    
    # Show expected impact
    impact = data['expected_impact']
    print(f"\n Expected Impact:")
    print(f"  GHG Reduction: {impact['expected_greenhouse_gas_reduction']}")
    print(f"  Biodiversity Benefit: {impact['biodiversity_benefit']}")
    print(f"  Timeline to Results: {impact['timeline_to_measurable_results']}")
else:
    print(f" Error: {policy_result.error}")

## 7. Policy Recommendation Agent

Generate evidence-based climate interventions and policy recommendations.

In [ ]:
from agents.reasoning_agent import ReasoningAgent

print("Initializing Reasoning Agent...")

reasoning_agent = ReasoningAgent(config.get('agents.reasoning', {}))

# Different types of reasoning questions
questions = [
    "Why is deforestation increasing in Ireland?",
    "What is the trend in emissions?",
    "How is climate change affecting biodiversity?"
]

print(f"\nProcessing {len(questions)} reasoning questions:\n")

for i, question in enumerate(questions, 1):
    print(f"{i}. Question: {question}")
    
    result = reasoning_agent.run({
        'question': question,
        'context': {
            'satellite_data': emissions_result.data if emissions_result.success else {},
            'climate_data': climate_data if climate_result.success else {},
            'region': 'Ireland'
        }
    })
    
    if result.success:
        data = result.data
        print(f"   Type: {data.get('question_type', 'general')}")
        print(f"   Explanation: {data.get('explanation', 'N/A')[:80]}...")
        print(f"   Confidence: {data.get('confidence', 0):.0%}\n")
    else:
        print(f"   Error: {result.error}\n")

## 6. LLM-Powered Reasoning Agent

Use the Reasoning Agent to explain climate phenomena and answer complex questions.

In [ ]:
from utils import VectorStoreManager

print("Setting up FAISS Vector Database...")

# Create vector store manager
vector_store = VectorStoreManager(
    index_path='./data/faiss_index',
    embedding_model='sentence-transformers/all-MiniLM-L6-v2'
)

# Create mock embeddings for climate information
climate_documents = [
    "Ireland's emissions increased by 5% in 2024 compared to 2023",
    "Deforestation in Irish forests detected in satellite imagery",
    "Biodiversity index in Irish wetlands declining over past 3 years",
    "Renewable energy installations reduced emissions by 12%",
    "Agriculture sector contributes 35% of total emissions",
    "Dublin region shows urban expansion into agricultural areas",
    "Temperature anomalies recorded in western Ireland",
    "Habitat loss threatens native bird populations"
]

# Create mock embeddings (in production would use actual embeddings)
mock_embeddings = [
    np.random.randn(384).tolist() for _ in climate_documents
]

# Add to vector store
print(f"Adding {len(climate_documents)} documents to vector database...")
vector_store.add_documents(climate_documents, mock_embeddings)

print(f" Vector store created with {len(climate_documents)} documents")

# Example similarity search
query_embedding = np.random.randn(384).tolist()
print(f"\n Example similarity search (top 3 results):")
results = vector_store.search(query_embedding, top_k=3)
for i, (distance, doc) in enumerate(results, 1):
    print(f"  {i}. (dist={distance:.3f}) {doc[:60]}...")

## 5. Vector Database Setup with FAISS

Store embeddings for semantic search across climate data.

In [ ]:
from agents.data_agent import DataAgent
from data import ClimateDataProcessor, EmissionsProcessor, BiodiversityAnalyzer

print(" Initializing Data Agent...")

data_agent = DataAgent(config.get('agents.data', {}))

# Request emissions data
print("\n Fetching Emissions Data for Ireland (2020-2024)...")
emissions_result = data_agent.run({
    'data_type': 'emissions',
    'region': 'Ireland',
    'temporal_range': [2020, 2024]
})

if emissions_result.success:
    emissions_data = emissions_result.data
    print(f" Emissions data retrieved:")
    print(f"  Region: {emissions_data['region']}")
    print(f"  GHGs tracked: {list(emissions_data['emissions'].keys())}")
    
    # Analyze CO2 trend
    co2_data = emissions_data['emissions']['CO2']
    print(f"\n  CO2 Analysis:")
    print(f"    Years: {co2_data['years']}")
    print(f"    Trend: {co2_data['trend']} ({co2_data['mean']:.1f} ± {co2_data['std_dev']:.1f})")

# Request climate data
print("\nFetching Climate Data for Ireland...")
climate_result = data_agent.run({
    'data_type': 'climate',
    'region': 'Ireland',
    'temporal_range': [2020, 2024]
})

if climate_result.success:
    climate_data = climate_result.data
    print(f" Climate data retrieved:")
    temp_data = climate_data['temperature']
    precip_data = climate_data['precipitation']
    print(f"  Temperature: {temp_data['mean']:.1f}°C ± {temp_data['std_dev']:.1f}°C")
    print(f"  Precipitation: {precip_data['mean']:.1f}mm (trend: {precip_data['trend']})")

# Request biodiversity data
print("\n Fetching Biodiversity Data...")
bio_result = data_agent.run({
    'data_type': 'biodiversity',
    'region': 'Ireland',
    'temporal_range': [2020, 2024]
})

if bio_result.success:
    bio_data = bio_result.data
    print(f"Biodiversity data retrieved:")
    print(f"  Biodiversity Index: {bio_data['biodiversity_index']:.2f}/1.0")
    print(f"  Species by type:")
    for species_type, count in bio_data['species_count'].items():
        print(f"    {species_type}: {count}")
    print(f"  Threatened Species:")
    for level, count in bio_data['threatened_species'].items():
        print(f"    {level}: {count}")

## 4. Climate Dataset Integration

Aggregate emissions, biodiversity, and climate data from multiple sources.

In [ ]:
from agents.satellite_agent import SatelliteAgent

print("Initializing Satellite Agent...")

# Create agent with config
sat_agent = SatelliteAgent(config.get('agents.satellite', {}))

# Prepare input - in production would use real satellite data
satellite_input = {
    'image_array': rgb * 255,  # Convert RGB to 0-255 range
    'location': 'Dublin, Ireland',
    'timestamp': '2024-03-27'
}

# Run agent
print("Running land-use classification...")
result = sat_agent.run(satellite_input)

if result.success:
    print(f"\nClassification Result:")
    classification = result.data['classification']
    print(f"  Land Use Class: {classification['land_use_class']}")
    print(f"  Confidence: {classification['confidence']:.2%}")
    print(f"\n  Class Probabilities:")
    for land_class, prob in list(classification['all_probabilities'].items())[:5]:
        print(f"    {land_class}: {prob:.2%}")
    
    print(f"\n Explainability (Grad-CAM):")
    print(f"  {result.data['explainability']}")
else:
    print(f" Error: {result.error}")

## 3. Land-Use Classification with CNN

Use the Satellite Agent to classify land-use types from satellite imagery.

In [ ]:
from geospatial import SentinelDataHandler, GeospatialProcessor

# Create mock Sentinel-2 bands (in production, would load from GEE or files)
print("Creating mock Sentinel-2 data...")

# Simulate 10m resolution bands (Red, Green, Blue, NIR)
h, w = 256, 256
np.random.seed(42)

mock_bands = {
    'B2': np.random.randint(100, 200, (h, w), dtype=np.uint16),  # Blue
    'B3': np.random.randint(120, 220, (h, w), dtype=np.uint16),  # Green
    'B4': np.random.randint(80, 180, (h, w), dtype=np.uint16),   # Red
    'B8': np.random.randint(150, 250, (h, w), dtype=np.uint16),  # NIR
    'B11': np.random.randint(100, 200, (h, w), dtype=np.uint16), # SWIR
}

print(f"Created Sentinel-2 dataset: {h}x{w}px, {len(mock_bands)} bands")

# Calculate NDVI (Normalized Difference Vegetation Index)
print("\n Calculating spectral indices...")
ndvi = SentinelDataHandler.calculate_index(mock_bands, 'NDVI')
ndbi = SentinelDataHandler.calculate_index(mock_bands, 'NDBI')

print(f"  NDVI range: [{ndvi.min():.3f}, {ndvi.max():.3f}]")
print(f"  NDBI range: [{ndbi.min():.3f}, {ndbi.max():.3f}]")

# Create RGB composite
rgb = SentinelDataHandler.rgb_composite(mock_bands)
print(f"  RGB composite shape: {rgb.shape}")

# Geospatial utilities
print("\nGeospatial Processing:")
bbox = GeospatialProcessor.create_bbox(53.3498, -6.2603, side_length_km=10)
print(f"  Bounding box around Dublin (-10km to +10km):")
print(f"    Lat: [{bbox['min_lat']:.4f}, {bbox['max_lat']:.4f}]")
print(f"    Lon: [{bbox['min_lon']:.4f}, {bbox['max_lon']:.4f}]")

## 2. Satellite Data Acquisition & Preprocessing

Load and prepare Sentinel-2 satellite imagery with spectral indices.

In [ ]:
# Load Configuration
config = Config(Path.cwd().parent / 'config' / 'settings.yaml')

print(" Loaded Configuration:")
print(f"  LLM Provider: {config.get('llm.provider', 'Not configured')}")
print(f"  Database: {config.get('database.type', 'Not configured')}")
print(f"  Cache Enabled: {config.get('cache.enabled', True)}")
print(f"  Logging Level: {config.get('logging.level', 'INFO')}")

# Test configuration
config_dict = config.to_dict()
print(f"\n Configuration loaded with {len(config_dict)} sections")

## 1. Environment Setup & Library Imports

Configure APIs, load credentials, and initialize the Climate Intelligence system.

In [ ]:
# Environment Setup & Library Imports
import sys
from pathlib import Path

# Add src to path for imports
sys.path.insert(0, str(Path.cwd().parent / 'src'))

# Core imports
import numpy as np
import pandas as pd
import json
from datetime import datetime, timedelta
import warnings
warnings.filterwarnings('ignore')

# Setup logging
from utils import Logger, Config
Logger.setup(log_level='INFO')

print("Climate Intelligence Copilot initialized")

# Climate Intelligence Copilot
## AI-Powered Climate Decision Support with Satellite Data & LLMs

This notebook demonstrates the full Climate Intelligence Copilot system:
- Satellite imagery analysis (Sentinel-2)
- Climate data integration
- LLM-powered reasoning
- Policy recommendations
- Multi-agent orchestration

Let's explore how different agents work together to answer complex climate questions!